In [7]:
# Cryptocurrency Liquidity Prediction - Final Version

# =========================
# 1. IMPORT LIBRARIES
# =========================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

# =========================
# 2. LOAD DATA
# =========================
print("Loading dataset...")
df = pd.read_csv(r"C:\Users\Pratham Patidar\Downloads\coin_gecko_2022-03-16.csv")

print("\nFirst 5 rows:")
print(df.head())

# =========================
# 3. BASIC EDA
# =========================
print("\nDataset Info:")
print(df.info())

print("\nStatistical Summary:")
print(df.describe())

print("\nMissing Values:")
print(df.isnull().sum())

# =========================
# 4. DATA CLEANING
# =========================
df = df.ffill()  # forward fill missing values

# =========================
# 5. FEATURE ENGINEERING
# =========================
print("\nCreating new features...")

# Target variable: Liquidity
df['liquidity'] = df['24h_volume'] / df['mkt_cap']

# Volatility
df['volatility'] = df['price'].pct_change().fillna(0)

# Moving Average
df['moving_avg_price'] = df['price'].rolling(window=7).mean().bfill()

# Log Transform (reduce skewness)
df['log_volume'] = np.log1p(df['24h_volume'])
df['log_mkt_cap'] = np.log1p(df['mkt_cap'])

# =========================
# 6. VISUALIZATION
# =========================
plt.figure(figsize=(8,5))
sns.histplot(df['price'], kde=True)
plt.title("Price Distribution")
plt.savefig("price_distribution.png")
plt.close()

numeric_df = df.select_dtypes(include=[np.number])

plt.figure(figsize=(10,6))
sns.heatmap(numeric_df.corr(), annot=False, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.savefig("correlation_matrix.png")
plt.close()

# =========================
# 7. FEATURE SELECTION
# =========================
features = ['price', 'log_volume', 'log_mkt_cap', 'volatility', 'moving_avg_price']
target = 'liquidity'

X = df[features]
y = df[target]

# =========================
# 8. TRAIN TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 9. MODEL TRAINING
# =========================
print("\nTraining Random Forest model...")

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

model.fit(X_train, y_train)

# =========================
# 10. PREDICTION
# =========================
y_pred = model.predict(X_test)

# =========================
# 11. EVALUATION
# =========================
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\nFinal Model Performance:")
print(f"MAE: {mae}")
print(f"RMSE: {rmse}")
print(f"R2 Score: {r2}")

# =========================
# 12. SAMPLE OUTPUT
# =========================
results = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred
})

print("\nSample Predictions:")
print(results.head())

# =========================
# 13. SAVE MODEL
# =========================
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

print("\nModel saved as model.pkl")

# =========================
# DONE
# =========================
print("\nProject completed successfully!")

Loading dataset...

First 5 rows:
       coin symbol         price     1h    24h     7d    24h_volume  \
0   Bitcoin    BTC  40859.460000  0.022  0.030  0.055  3.539076e+10   
1  Ethereum    ETH   2744.410000  0.024  0.034  0.065  1.974870e+10   
2    Tether   USDT      1.000000 -0.001 -0.001  0.000  5.793497e+10   
3       BNB    BNB    383.430000  0.018  0.028  0.004  1.395854e+09   
4  USD Coin   USDC      0.999874 -0.001  0.000 -0.000  3.872274e+09   

        mkt_cap        date  
0  7.709915e+11  2022-03-16  
1  3.271044e+11  2022-03-16  
2  7.996516e+10  2022-03-16  
3  6.404382e+10  2022-03-16  
4  5.222214e+10  2022-03-16  

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   coin        500 non-null    object 
 1   symbol      500 non-null    object 
 2   price       500 non-null    float64
 3   1h          497 non-null    fl